In [17]:
import numpy as np
import pandas as pd
from rpy2.robjects import r, conversion, pandas2ri
from helper_functions import normalize_household_data, dict_to_named_list

In [2]:
pandas2ri.activate()
r.source('Simulator/Simulator.R')
model_r = r['simulate_and_reformat']

R[write to console]: 
Attaching package: ‘dplyr’


R[write to console]: The following objects are masked from ‘package:stats’:

    filter, lag


R[write to console]: The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


R[write to console]: 
Attaching package: ‘actuar’


R[write to console]: The following objects are masked from ‘package:stats’:

    sd, var


R[write to console]: The following object is masked from ‘package:grDevices’:

    cm




In [3]:
def prior(batch_size: int) -> np.ndarray:
    param_batch = np.zeros((batch_size, 12))
    # alpha
    param_batch[:, 0] = np.random.uniform(0, 0.1, batch_size)
    # beta
    param_batch[:, 1] = np.random.uniform(0, 10, batch_size)
    # delta
    param_batch[:, 2] = np.random.uniform(-3, 3, batch_size)
    # mu_inf
    param_batch[:, 3:8] = np.exp(np.random.normal(0, 1, (batch_size, 5)))
    # mu_susc
    param_batch[:, 8:10] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    # mu_protect
    param_batch[:, 10:] = np.exp(np.random.normal(0, 1, (batch_size, 2)))
    return param_batch

test = prior(1).flatten()

In [4]:
param_names = ['alpha', 'beta', 'delta', 
               'mu_inf_SC', 'mu_inf_SA', 'mu_inf_AI', 'mu_inf_AC', 'mu_inf_AA',
               'mu_susc_C', 'mu_susc_A', 
               'mu_protect_acq', 'mu_protect_transm']

In [5]:
def simulator(params: np.ndarray,
              variant: str = "alpha",
              selection_procedure: str = "pedcov") -> np.ndarray:
    """
    Simulate data with given parameters and reformat it to a numpy array.
    :param params: parameters for the simulation
    :param variant: variant of the virus (alpha or omicron)
    :param selection_procedure: selection procedure for the households (pedcov or random)
    :return: simulated data as numpy array
    """
    # create dict from params and param_names
    par_dict = dict(zip(param_names, params))
    # update dict with fixed parameters
    par_dict.update({'variant': variant, 'selection_procedure': selection_procedure})

    # minimal_length should be the maximal length of the time series in the real data set
    if par_dict['variant'] == "alpha":
        minimal_length = 8
    elif par_dict['variant'] == "omicron":
        minimal_length = 9
    else:
        raise ValueError("Variant not supported. Must be 'alpha' or 'omicron'.")

    # simulate data
    sim_data_r = model_r(dict_to_named_list(par_dict))
    # convert to pandas dataframe
    sim_data_full = conversion.rpy2py(sim_data_r)
    print(sim_data_full)
    # normalize data and return as numpy array
    #sim_data_norm = normalize_household_data(sim_data_full, minimal_length=minimal_length, use_one_hot_encoding=True)
    return sim_data_full
#model_py.__name__ = 'simulate_and_reformat'

In [6]:
sim_data = simulator(test)

     id_patient       id_hh  hh_size  date_sympt  infect_status  end_followup  \
1         14545  25-a-28-p1        3        32.0            2.0          92.0   
2         14546  25-a-28-p1        3        35.0            1.0          92.0   
3         14547  25-a-28-p1        3        36.0            1.0          92.0   
4          6257  90-a-12-p1        2        32.0            1.0          97.0   
5          6258  90-a-12-p1        2        40.0            1.0          97.0   
..          ...         ...      ...         ...            ...           ...   
522       16680  24-a-32-p1        5        37.0            1.0         129.0   
523       16681  24-a-32-p1        5        36.0            1.0         129.0   
524       16682  24-a-32-p1        5        33.0            1.0         129.0   
525       16683  24-a-32-p1        5        34.0            2.0         129.0   
526       16684  24-a-32-p1        5        35.0            1.0         129.0   

     age  protected variant

In [10]:
# extract the old household id  (number before "-a")
sim_data['id_hh_old'] = sim_data['id_hh'].str.extract(r'(\d+)-a')

In [32]:
old_ids = sim_data.groupby(['id_hh', 'id_hh_old']).size()
# count the number of different old ids in index
old_ids.index.get_level_values(1).value_counts()

id_hh_old
25     4
57     4
123    4
117    3
115    3
      ..
36     1
37     1
39     1
40     1
99     1
Name: count, Length: 81, dtype: int64

In [ ]:
# columns: date_sympt_norm, infect_status_norm, age_norm, protected
# rows: empty individuals in the beginning (households are of same size)
# last row: end_followup_norm with 1

In [ ]:
# load a presimulation using pickle
import pickle
with open('presimulations/presim_file_1.pkl', 'rb') as f:
    presim = pickle.load(f)
    
with open('presimulations/presim_file_2.pkl', 'rb') as f:
    presim_2 = pickle.load(f)

In [ ]:
presim[0]['prior_draws']#-presim_2[5]['prior_draws']

In [ ]:
len(presim)

In [ ]:
presim_2[0]['prior_draws']#[:, :, :, 0]*1000

In [ ]:
with open('valid_data.pickle', 'rb') as f:
    valid_data = pickle.load(f)

In [ ]:
valid_data['prior_draws']